[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CS7150/classdemos/blob/main/optimization/meanrms.ipynb)

# Mean vs. RMS: the diagnostic behind Adam

Adam decides, separately for every parameter, how confidently to step. It does this by comparing
two exponential moving averages (EMAs) of the gradient stream $g_1, g_2, \dots$ for that parameter:

$$\mathrm{mean}_t = \mathrm{EMA}_\beta(g)_t = \beta \, \mathrm{mean}_{t-1} + (1-\beta)\, g_t$$

$$\mathrm{RMS}_t = \sqrt{\mathrm{EMA}_\beta(g^2)_t}, \qquad \mathrm{EMA}_\beta(g^2)_t = \beta \, \mathrm{EMA}_\beta(g^2)_{t-1} + (1-\beta)\, g_t^2$$

These are the *same* recurrence, run on two different inputs: `mean` averages the raw, signed
gradient $g_t$, while `RMS` averages the squared gradient $g_t^2$ and then takes a square root.
That one difference &mdash; squaring before averaging &mdash; changes what the average is sensitive to.

Think about what happens when the gradient keeps flipping sign, say $g_t \approx \pm 1$ at random.
In the `mean` recurrence, a $+1$ term and a $-1$ term nearly cancel, so as more terms accumulate,
$\mathrm{mean}_t$ is pulled toward $0$: the moving average of a sign-alternating sequence collapses.
But in the `RMS` recurrence every term entering the average is $g_t^2 \approx 1$, positive regardless
of the sign of $g_t$ &mdash; squaring destroys the sign before the averaging ever sees it, so nothing
cancels, and $\mathrm{RMS}_t$ stays near $1$ even though `mean` has gone to $0$.

That gap between $\mathrm{mean}_t$ and $\mathrm{RMS}_t$ *is* the diagnostic. Adam's update looks at
the ratio $\mathrm{mean}_t / \mathrm{RMS}_t$ (in bias-corrected form, $\hat m_t/\sqrt{\hat v_t}$) for
each parameter: when the gradient has been pointing consistently one way, mean and RMS are close in
magnitude, the ratio is near $\pm 1$, and Adam takes a confident, full-size step in that direction.
When the gradient has been flipping sign &mdash; a noisy or oscillating direction &mdash; mean is much
smaller than RMS, the ratio shrinks toward $0$, and Adam automatically takes a smaller step there.
RMS in the denominator is exactly what lets Adam tell "a large, consistent gradient" apart from
"a large but noisy, sign-flipping gradient," even though both would look identical to a plain
running average of $|g_t|$.

The two cells below build a synthetic gradient stream, run both recurrences on it side by side, and
plot the result &mdash; first for an oscillating stream (mean collapses, RMS doesn't), then for a
consistent-sign stream (mean tracks RMS closely) &mdash; so you can see this diagnostic appear directly
in the numbers.

See the live interactive version: https://cs7150.github.io/classdemos/demos/optimizers/meanrms.html

In [ ]:
#@title Setup: gradient stream generator + plotting helper (double-click to inspect) { display-mode: "form" }
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def make_gradient_stream(steps=400, drift=0.0, omega=1.2, noise=0.4, seed=0):
    """A synthetic noisy, sign-flipping gradient stream, same shape as the live demo:
    g(t) = drift + sin(t * omega) + noise * uniform(-1, 1).
    `omega` controls how often the sign flips; `drift` biases it consistently one way.
    """
    rng = np.random.RandomState(seed)
    t = np.arange(steps)
    return drift + np.sin(t * omega) + noise * rng.uniform(-1, 1, size=steps)

def plot_mean_vs_rms(g, mean_hist, rms_hist, title):
    fig, ax = plt.subplots(figsize=(9, 4))
    t = np.arange(len(g))
    ax.bar(t, g, width=0.8, color=np.where(g >= 0, '#4a90d9', '#2a5a8a'), alpha=0.7, label='g (raw gradient)')
    ax.plot(t, mean_hist, color='#0F6E56', linewidth=2, label='mean = EMA(g)')
    ax.plot(t, rms_hist, color='#BA7517', linewidth=2, linestyle='--', label='RMS = sqrt(EMA(g^2))')
    ax.plot(t, -np.array(rms_hist), color='#BA7517', linewidth=2, linestyle='--')
    ax.axhline(0, color='#888780', linewidth=1)
    ax.set_title(title)
    ax.set_xlabel('step t')
    ax.legend(loc='upper right')
    plt.show()

## The core algorithm

Both `mean` and `rms` are computed with the *same* EMA recurrence &mdash; the only difference
is what we feed it: the raw gradient `g`, or its square `g**2`.

In [ ]:
beta = 0.95
g_stream = make_gradient_stream(steps=400, drift=0.0, omega=1.2, noise=0.4)

mean = 0.0
msq = 0.0
mean_hist, rms_hist = [], []

for g in g_stream:
    mean = beta * mean + (1 - beta) * g       # EMA of g
    msq  = beta * msq  + (1 - beta) * g ** 2  # EMA of g^2
    rms = np.sqrt(msq)
    mean_hist.append(mean)
    rms_hist.append(rms)

plot_mean_vs_rms(g_stream, mean_hist, rms_hist, 'Oscillating gradient: mean collapses, RMS does not')

## Compare: a consistent-sign gradient

Give the stream a strong drift so the gradient rarely changes sign. Now `mean` tracks `RMS`
closely &mdash; Adam sees this as a safe, consistent direction and takes full-size steps.

In [ ]:
g_stream2 = make_gradient_stream(steps=400, drift=0.9, omega=0.0, noise=0.3)

mean = 0.0
msq = 0.0
mean_hist2, rms_hist2 = [], []

for g in g_stream2:
    mean = beta * mean + (1 - beta) * g
    msq  = beta * msq  + (1 - beta) * g ** 2
    rms = np.sqrt(msq)
    mean_hist2.append(mean)
    rms_hist2.append(rms)

plot_mean_vs_rms(g_stream2, mean_hist2, rms_hist2, 'Consistent-sign gradient: mean tracks RMS closely')

print(f"oscillating: |mean|/RMS (final) = {abs(mean_hist[-1]) / rms_hist[-1]:.3f}  (small -> Adam slows down)")
print(f"consistent:  |mean|/RMS (final) = {abs(mean_hist2[-1]) / rms_hist2[-1]:.3f}  (near 1 -> Adam moves fast)")